In [ ]:
"""
Notebook: base_analitica_setorial_v3.py
Projeto: Brasil Público — Escala 6x1
Etapa: 03_modelagem
Versão: 3 — ajustes metodológicos e de nomenclatura

Mudanças em relação à v2:
  - "hipótese nula" → "cenário base"
  - Faixa de incerteza explicitada [-0,513%, -0,77%]
  - Normalização do impacto pelo VAB setorial (intensidade relativa)
  - Anotações de qualidade da proxy por setor
  - Limitações documentadas inline
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE      = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN  = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN    = BASE / "data/processed/pib_setorial_2012_2023.csv"
BASE_OUT  = BASE / "data/processed/base_analitica_setorial_2023.csv"
TAB_OUT   = BASE / "outputs/tables/cenario_base_setorial.csv"
DIR_FIGS  = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

print(f"PIB total 2023: R$ {pib_total_rs/1e12:.2f} trilhões")
print(f"\nFaixas corrigidas (amostra):")
print(df_horas[["n", "media", "pct_abaixo_40", "pct_40h_exatas",
                "pct_41_44h", "pct_acima_44h"]].to_string())

# Validação: somas devem ser ~100%
assert (df_horas["pct_abaixo_40"] + df_horas["pct_40h_exatas"] +
        df_horas["pct_41_44h"] + df_horas["pct_acima_44h"]).between(99, 101).all(), \
    "ERRO: somas das faixas fora de 100%"
print("\n✓ Validação das faixas: OK")

In [ ]:
H_MEDIA_AFETADOS = 42.0
H_PROPOSTA       = 40.0
REDUCAO_AFETADOS = (H_PROPOSTA - H_MEDIA_AFETADOS) / H_MEDIA_AFETADOS
SEMANAS_ANO      = 52

print(f"\nParâmetros do modelo:")
print(f"  Faixa afetada: 41–44h (média assumida: {H_MEDIA_AFETADOS:.0f}h)")
print(f"  Redução para afetados: {H_MEDIA_AFETADOS:.0f}h → {H_PROPOSTA:.0f}h = {REDUCAO_AFETADOS*100:.2f}%")
print(f"  Hipótese: produtividade/hora constante (cenário base)")

correspondencia = [
    # (setor_pnad, col_cnt, incluir, qualidade_proxy)
    ("Agropecuária",          "agropecuaria",          True,  "direta"),
    ("Indústria geral",       "ind_transformacao",     True,  "parcial"),
    ("Construção",            "construcao",            True,  "direta"),
    ("Comércio e rep.",       "comercio",              True,  "direta"),
    ("Transp. e armaz.",      "transporte",            True,  "direta"),
    ("Inf. e serv. prof.",    "info_comunicacao",      True,  "parcial"),
    ("Adm. públ. e educação", "adm_publica_saude_educ",True, "ampla"),
    ("Alojamento e alim.",    "outros_servicos",       False, "muito_imperfeita"),
    ("Saúde",                 "adm_publica_saude_educ",False, "nao_separavel"),
    ("Serv. domésticos",      "outros_servicos",       False, "muito_imperfeita"),
]

In [ ]:
registros = []

for setor, col, incluir, qualidade in correspondencia:
    if setor not in df_horas.index:
        continue

    n     = df_horas.loc[setor, "n"]
    h_med = df_horas.loc[setor, "media"]
    h_mdn = df_horas.loc[setor, "mediana"]
    p4144 = df_horas.loc[setor, "pct_41_44h"]
    p_44  = df_horas.loc[setor, "pct_acima_44h"]
    p_40  = df_horas.loc[setor, "pct_40h_exatas"]
    p_sub = df_horas.loc[setor, "pct_abaixo_40"]

    vab_rs          = pib_2023[col] * 1e6
    total_horas_ano = n * h_med * SEMANAS_ANO
    produt_hora     = vab_rs / total_horas_ano

    parcela_afetada   = p4144 / 100.0
    delta_horas_setor = parcela_afetada * REDUCAO_AFETADOS
    delta_vab_rs      = vab_rs * delta_horas_setor

    # Intensidade: impacto relativo ao VAB do próprio setor
    intensidade_pct = delta_horas_setor * 100

    registros.append({
        "setor":             setor,
        "col_cnt":           col,
        "qualidade_proxy":   qualidade,
        "incluir":           incluir,
        "n_ocupados":        int(n),
        "h_media":           round(h_med, 2),
        "h_mediana":         round(h_mdn, 1),
        "pct_abaixo_40":     round(p_sub, 1),
        "pct_40h_exatas":    round(p_40, 1),
        "pct_41_44h":        round(p4144, 2),
        "pct_acima_44h":     round(p_44, 1),
        "vab_r$_bi":         round(vab_rs / 1e9, 2),
        "produt_r$_hora":    round(produt_hora, 2),
        "parcela_afetada":   round(parcela_afetada, 4),
        "delta_vab_r$_bi":   round(delta_vab_rs / 1e9, 3),
        "intensidade_pct":   round(intensidade_pct, 3),
    })

df_base = pd.DataFrame(registros)

print("\n── BASE ANALÍTICA SETORIAL 2023 ─────────────────────────")
cols_view = ["setor", "qualidade_proxy", "h_media", "pct_41_44h",
             "vab_r$_bi", "produt_r$_hora", "delta_vab_r$_bi", "intensidade_pct"]
print(df_base[cols_view].to_string(index=False))

In [ ]:
df_incl = df_base[df_base["incluir"] == True].copy()
df_excl = df_base[df_base["incluir"] == False].copy()

delta_incl = df_incl["delta_vab_r$_bi"].sum() * 1e9
delta_excl = df_excl["delta_vab_r$_bi"].sum() * 1e9
delta_total = delta_incl + delta_excl

pct_incl  = delta_incl / pib_total_rs * 100
pct_excl  = delta_excl / pib_total_rs * 100
pct_total = delta_total / pib_total_rs * 100

print("\n══ CENÁRIO BASE — sem ajuste de produtividade ══════════")
print(f"Hipótese: P_i constante após redução de jornada")
print(f"Faixa afetada: trabalhadores com 41h < H ≤ 44h")
print(f"Redução assumida: {H_MEDIA_AFETADOS:.0f}h → {H_PROPOSTA:.0f}h ({REDUCAO_AFETADOS*100:.2f}%)")
print()
print(f"Setores incluídos (proxies diretas/parciais):")
print(f"  ΔVAB = R$ {delta_incl/1e9:.2f} bi  →  ΔPIB = {pct_incl:.3f}%")
print()
print(f"Setores excluídos (proxies muito imperfeitas):")
print(f"  ΔVAB estimado = R$ {delta_excl/1e9:.2f} bi  →  ΔPIB implícito = {pct_excl:.3f}%")
print()
print(f"── FAIXA DE INCERTEZA DO MODELO ────────────────────────")
print(f"  Limite inferior (só setores incluídos):  {pct_incl:.3f}%")
print(f"  Limite superior (todos os setores):      {pct_total:.3f}%")
print(f"  Referência CNI:                          -0,700%")
print()
print(f"Detalhamento (setores incluídos):")
print(df_incl[["setor", "pct_41_44h", "delta_vab_r$_bi", "intensidade_pct"]]
      .sort_values("delta_vab_r$_bi").to_string(index=False))

In [ ]:
print("\n── INTENSIDADE RELATIVA (impacto / VAB próprio) ─────────")
print("Mede o quanto cada setor é internamente afetado")
print(df_incl[["setor", "h_media", "pct_41_44h", "intensidade_pct"]]
      .sort_values("intensidade_pct").to_string(index=False))

In [ ]:
df_base.to_csv(BASE_OUT, index=False)
df_incl.to_csv(TAB_OUT, index=False)
print(f"\n✓ Base analítica salva: {BASE_OUT}")
print(f"✓ Tabela cenário base salva: {TAB_OUT}")

In [ ]:
df_g = df_incl.sort_values("delta_vab_r$_bi", ascending=True).copy()

qualidade_cores = {
    "direta":  "#2C6FBF",
    "parcial": "#F5A623",
    "ampla":   "#E84545",
}
cores_barra = [qualidade_cores.get(q, "#888") for q in df_g["qualidade_proxy"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Painel 1: impacto absoluto (R$ bi)
bars1 = ax1.barh(df_g["setor"], df_g["delta_vab_r$_bi"], color=cores_barra, height=0.6)
ax1.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars1, df_g.iterrows()):
    ax1.text(
        row["delta_vab_r$_bi"] - 0.1,
        bar.get_y() + bar.get_height() / 2,
        f"R$ {row['delta_vab_r$_bi']:.2f} bi",
        va="center", ha="right", fontsize=8.5
    )

ax1.set_xlabel("Variação no VAB (R$ bilhões)", fontsize=10)
ax1.set_title("Impacto absoluto\n(contribuição para ΔPIB)", fontsize=11, loc="left")

# Painel 2: intensidade relativa (% do VAB próprio)
df_g2 = df_incl.sort_values("intensidade_pct", ascending=True).copy()
cores2 = [qualidade_cores.get(q, "#888") for q in df_g2["qualidade_proxy"]]

bars2 = ax2.barh(df_g2["setor"], df_g2["intensidade_pct"], color=cores2, height=0.6)

for bar, (_, row) in zip(bars2, df_g2.iterrows()):
    ax2.text(
        row["intensidade_pct"] - 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{row['intensidade_pct']:.3f}%",
        va="center", ha="right", fontsize=8.5
    )

ax2.set_xlabel("Variação relativa no VAB próprio (%)", fontsize=10)
ax2.set_title("Intensidade relativa\n(impacto / VAB do próprio setor)", fontsize=11, loc="left")

# Legenda de qualidade da proxy
legenda = [
    mpatches.Patch(color="#2C6FBF", label="Proxy direta"),
    mpatches.Patch(color="#F5A623", label="Proxy parcial"),
    mpatches.Patch(color="#E84545", label="Proxy ampla"),
]
fig.legend(handles=legenda, loc="lower center", ncol=3, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, -0.05))

fig.suptitle(
    "Cenário base: impacto setorial da redução de jornada (44h→40h) — Brasil 2023\n"
    "Sem ajuste de produtividade · Faixa afetada: 41–44h · PNAD Contínua + CNT/IBGE",
    fontsize=12, y=1.01
)

fig.text(
    0.01, -0.08,
    f"ΔPIB (setores incluídos): {pct_incl:.3f}%  |  "
    f"Faixa de incerteza: [{pct_incl:.3f}%, {pct_total:.3f}%]  |  "
    f"Referência CNI: -0,700%  |  "
    "Limitação: modelo de 1ª ordem, sem efeitos de segunda ordem.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "11_cenario_base_setorial.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 11 salvo.")